# 📄📂📄📂 <span style="color: white; background-color: Goldenrod"><b> Expurgo de Arquivos </b></span></p>

🧩 <span style="color: Gold"><b> 1- Controle e Governança do Processo </b></span></p>
- A automação registra todas as etapas no arquivo PROCESSOS.xlsx
- Com logs contendo:
    - ID do processo  
    - Data/hora  
- Isso cria trilha de auditoria, permitindo rastreamento total do expurgo

📥 <span style="color: Gold"><b> 2- Leitura dos Arquivos da Pasta de Expurgo </b></span></p>
- O processo acessa o diretório 2. ARQUIVOS MOVIDOS
- Lista todos os arquivos disponíveis  
- Ignora subpastas  
- Salva o total de arquivos antes do expurgo  
- Cada arquivo recebe seu:
    - nome  
    - data de última modificação
- Esses dados alimentam a base de decisão

📅 <span style="color: Gold"><b> 3- Cálculo da Janela de Retenção (30 dias) </b></span></p>
- O script define data_limite = hoje – 30 dias
- Compara a data de modificação de cada arquivo  
- Seleciona apenas arquivos com mais de 30 dias  
- Cria a lista final de expurgo  

🗑️ <span style="color: Gold"><b> 4- Execução do Expurgo (Exclusão Física) </b></span></p>
- Faz a exclusão com os.remove()  
- Garante que apenas arquivos seguros sejam removidos  
- Evita exclusão acidental de pastas  
- Registra cada etapa no controle

📊 <span style="color: Gold"><b> 5- Resumo Final da Execução </b></span></p>
- O sistema apresenta ao final:
    - Tempo total de execução  
    - Quantidade de arquivos antes do expurgo  
    - Quantidade de arquivos efetivamente expurgados  
    - Quantidade de arquivos restantes

# Importação das Bibliotecas

In [ ]:
import os
import pandas as pd
from datetime import datetime, date
from openpyxl import Workbook, load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.worksheet.table import Table, TableStyleInfo

# Carregando Base de Controle de Processos

In [ ]:
# Carregamento da base de controle de processos

id = 15

path_registros_processos = r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\PROCESSOS.xlsx'

registros_processos = pd.read_excel(path_registros_processos, sheet_name="REGISTROS", engine='openpyxl')

wb_p = load_workbook(path_registros_processos)

ws_p = wb_p['REGISTROS']

# Controle de atualização de processo: Etapa 0

tempo_0 = [id, datetime.today(), 0]

ws_p.append(tempo_0)

wb_p.save(path_registros_processos)

# Inicializando Variáveis

In [ ]:
# Variável com o diretório dos arquivos de backup

diretorio = r'X:\Gestão de Pessoas\Analytics\03 - Bases\2. ARQUIVOS MOVIDOS'

# Lista os arquivos do diretório, garantindo filtrar apenas arquivos e ignorando pastas

arquivos = os.listdir(diretorio)

arquivos = [f for f in arquivos if os.path.isfile(os.path.join(diretorio, f))]

# Controle de atualização de processo: Etapa 1

tempo_1 = [id, datetime.today(), 1]

ws_p.append(tempo_1)

wb_p.save(path_registros_processos)

# Criando Base de Arquivos e Data da Última Modificação

In [ ]:
# looping para incluir data de última modificação

dados = []

for arquivo in arquivos:
    caminho_arquivo = os.path.join(diretorio, arquivo)
    ultima_modificacao = os.path.getmtime(caminho_arquivo)
    data_formatada = datetime.fromtimestamp(ultima_modificacao).strftime('%Y-%m-%d %H:%M:%S')
    dados.append({'nome_arquivo': arquivo, 'ultima_modificacao': data_formatada})
    
# Convertendo em dataframe

total_arquivos = pd.DataFrame(dados)

# Variável com o total de arquivos na pasta antes do expurgo

qtd_arquivos_antes = total_arquivos.shape[0]

# Controle de atualização de processo: Etapa 2

tempo_2 = [id, datetime.today(), 2]

ws_p.append(tempo_2)

wb_p.save(path_registros_processos)

# Filtrando Período a ser Expurgado

In [ ]:
# Converte a coluna de string para datetime para permitir operações matemáticas com datas
total_arquivos['ultima_modificacao'] = pd.to_datetime(total_arquivos['ultima_modificacao'])

# Define a data limite (Data de hoje zerando as horas - 30 dias)
data_limite = pd.Timestamp.now().normalize() - pd.Timedelta(days=30)

# Filtra os arquivos anteriores à data limite e copia para um novo DataFrame
expurgar = total_arquivos[total_arquivos['ultima_modificacao'] < data_limite].copy()

# Renomeia a coluna para manter a compatibilidade com o restante do código original
expurgar = expurgar.rename(columns={'nome_arquivo': 'Nome'})

qtd_arquivos_expurgados = expurgar.shape[0]

# Geração de uma lista dos arquivos a serem expurgados

lista_expurgo = expurgar['Nome'].tolist()

# Controle de atualização de processo: Etapa 3

tempo_3 = [id, datetime.today(), 3]

ws_p.append(tempo_3)

wb_p.save(path_registros_processos)

# Looping para Expurgo

In [ ]:
for file in lista_expurgo:
    file_expurgar = os.path.join(diretorio, file) # os.path.join é mais seguro que concatenar string com '\'
    os.remove(file_expurgar)
    
# Controle de atualização de processo: Etapa 4

tempo_4 = [id, datetime.today(), 4]

ws_p.append(tempo_4)

wb_p.save(path_registros_processos)

# Resumo de Finalização do Processo

In [ ]:
print('--------------------------------------------------------------------------------------------')
print('')
print('     ✅  Processo finalizado')
print('')
print('     ⏱️   Tempo de execução:')
print('')
print(f'   {tempo_4[1] - tempo_0[1]}')
print('')
print(f'   {qtd_arquivos_antes} arquivos.')
print('')
print(f'   {qtd_arquivos_expurgados} expurgados.')
print('')
print(f'   {qtd_arquivos_antes - qtd_arquivos_expurgados} restantes.')
print('')
print('--------------------------------------------------------------------------------------------')